<a href="https://colab.research.google.com/github/TalCordova/RMBA_SemB26_TC_SC/blob/main/Course_Project_Text_Mining_TC_SC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course Project - Text Mining Features

Research Methods for Business Anlytics Course, Semester B, 2026

* Tal Cordova, 203868187
* Saar Cohen, 213499312

In this notebook, we will prepare a model to predict the probability of a user having a positive interaction with the companie's product.

The output of this model will be a feature to the model that's deciding if we should contact a user.

## Mount Drive and Load Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

file_path_saar = '/content/drive/MyDrive/Research Methods for Business Analytics/text_training.csv'

file_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/TM Exercise/text_training.csv'

file_path = file_path_tal

df = pd.read_csv(file_path)

print('File loaded successfully. Here are the first 5 rows:')
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File loaded successfully. Here are the first 5 rows:


,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick,rating
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Fit

In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Reload data if not already in memory
if 'df' not in dir() or df is None:
    print("Reloading training data...")
    file_path = '/content/drive/MyDrive/Research Methods for Business Analytics/text_training.csv'
    df = pd.read_csv(file_path)
    print("Done.")

if 'X' not in dir() or 'y' not in dir():
    print("Recreating X and y...")
    X = df.drop(columns=['rating', 'ID', 'halo', 'safeti'])
    y = df['rating']
    print("Done.")

Recreating X and y...
Done.


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2


best_lr = Pipeline([
    ('select', SelectKBest(chi2, k=1500)),
    ('clf', LogisticRegression(C=0.5, penalty='l2', solver='saga', max_iter=5000))
])
best_lr.fit(X, y)

Pipeline(steps=[('select',
                 SelectKBest(k=1500,
                             score_func=<function chi2 at 0x7fab70b011c0>)),
                ('clf',
                 LogisticRegression(C=0.5, max_iter=5000, solver='saga'))])

## Predict

### Predict for Train

In [6]:
# Load rollout file
reviews_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_training.csv'
df_reviews_train = pd.read_csv(reviews_train)

df_reviews_train.head()

,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,believ,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick
0,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,40,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,54,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,84,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [11]:
# Save IDs in original order - DO NOT SORT OR SHUFFLE
reviews_train_ids = df_reviews_train['ID']

# Drop same columns as training
X_reviews_train = df_reviews_train.drop(columns=['ID', 'halo', 'safeti'])

In [13]:
predictions = best_lr.predict_proba(X_reviews_train)

# The predict_proba method returns probabilities for both classes (negative and positive).
# We are interested in the probability of the positive class (usually the second column).
# Extract the probabilities for the positive class.
positive_predictions = predictions[:, 1]

# Verify record count before saving
if len(positive_predictions) != len(df_reviews_train):
    print(f"ERROR: prediction count ({len(positive_predictions)}) does not match rollout count ({len(df_reviews_train)}). File not saved.")
else:
    output = pd.DataFrame({
        'ID': reviews_train_ids,
        'rating': positive_predictions
    })

    # Tal's file
    output_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_train_preds.csv'
    output.to_csv(output_path_tal, index=False)
    print(f"OK: Tal's file saved - {len(output)} predictions")

    print(output.head(10))

OK: Tal's file saved - 1499 predictions
    ID    rating
0    3  0.200771
1   40  0.359464
2   47  0.981491
3   54  0.006640
4   84  0.962683
5   92  0.696444
6   95  0.298339
7  113  0.262681
8  114  0.006567
9  207  0.001673


### Predict for Rollout

In [15]:
# Load rollout file
reviews_rollout = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_rollout.csv'
df_reviews_rollout = pd.read_csv(reviews_rollout)

df_reviews_rollout.head()

,ID,singl,nutti,wait,disappoint,spring,crisp,lol,origin,adult,...,believ,glutenfre,coco,finger,smoothi,junk,finicki,rice,natur,stick
0,20002,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,20009,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20026,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,20037,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,20046,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
df_reviews_rollout.shape

(1501, 2001)

In [20]:
# Save IDs in original order - DO NOT SORT OR SHUFFLE
reviews_rollout_ids = df_reviews_rollout['ID']

# Drop same columns as training
X_reviews_rollout = df_reviews_rollout.drop(columns=['ID', 'halo', 'safeti'])

In [21]:
predictions = best_lr.predict_proba(X_reviews_rollout)

# The predict_proba method returns probabilities for both classes (negative and positive).
# We are interested in the probability of the positive class (usually the second column).
# Extract the probabilities for the positive class.
positive_predictions = predictions[:, 1]

# Verify record count before saving
if len(positive_predictions) != len(df_reviews_rollout):
    print(f"ERROR: prediction count ({len(positive_predictions)}) does not match rollout count ({len(df_reviews_rollout)}). File not saved.")
else:
    output = pd.DataFrame({
        'ID': reviews_rollout_ids,
        'rating': positive_predictions
    })

    # Tal's file
    output_path_tal = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_rollout_preds.csv'
    output.to_csv(output_path_tal, index=False)
    print(f"OK: Tal's file saved - {len(output)} predictions")

    print(output.head(10))

OK: Tal's file saved - 1501 predictions
      ID        rating
0  20002  5.132756e-08
1  20009  8.160113e-02
2  20026  7.172129e-01
3  20037  9.841181e-02
4  20046  9.578428e-01
5  20047  8.881992e-01
6  20053  2.876338e-02
7  20069  2.278023e-01
8  20077  6.507048e-01
9  20124  1.208782e-02
